# DSA 210 — Final Submission: Music Listening Behavior & Weather
## Ali Deha Ozan | Spring 2026

---

### What's new vs. Phase 4 (per instructor feedback)
| Feedback Point | Action Taken |
|---|---|
| Add a non-linear model | ✅ Random Forest Regressor added and compared against Linear Regression |
| Check for non-linear relationships | ✅ Residual plots + feature importance analysis |
| Add day-of-week as a feature | ✅ `day_of_week`, `is_weekend`, `month` added to all models |


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report
from scipy import stats

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded.')


In [ ]:
URL = 'https://raw.githubusercontent.com/deha-ozan/DSA210projectDeha/main/music_weather_daily.csv'
daily = pd.read_csv(URL, parse_dates=['date'])

daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend']  = (daily['day_of_week'] >= 5).astype(int)
daily['month']       = daily['date'].dt.month

print(f'Shape: {daily.shape}')
print(f'Date range: {daily["date"].min().date()} → {daily["date"].max().date()}')


---
## 1. Day-of-Week Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
daily.boxplot(column='total_listen_min', by='day_of_week', ax=axes[0])
axes[0].set_xticklabels(dow_labels)
axes[0].set_title('Listening Duration by Day of Week')
axes[0].set_xlabel('Day of Week')
axes[0].set_ylabel('Total Listen (min)')
plt.sca(axes[0]); plt.title('Listening Duration by Day of Week')

means = daily.groupby('is_weekend')['total_listen_min'].mean()
axes[1].bar(['Weekday','Weekend'], means.values, color=['steelblue','coral'], edgecolor='black')
axes[1].set_title('Mean Listening: Weekday vs Weekend')
axes[1].set_ylabel('Mean Total Listen (min)')
for i, v in enumerate(means.values):
    axes[1].text(i, v+1, f'{v:.1f} min', ha='center', fontweight='bold')

plt.suptitle('')
plt.tight_layout()
plt.show()

stat, p = stats.mannwhitneyu(
    daily[daily['is_weekend']==1]['total_listen_min'].dropna(),
    daily[daily['is_weekend']==0]['total_listen_min'].dropna(),
    alternative='two-sided')
print(f'Mann-Whitney U — weekday vs weekend: U={stat:.1f}, p={p:.4f}')
print(f'→ {"Significant" if p < 0.05 else "Not significant"} (α=0.05)')


---
## 2. Feature Definitions

Hardcoded to avoid auto-detection errors:
- **Weather features:** `temp_mean`, `temp_max`, `precipitation`, `sunshine_hours`
- **Time features:** `day_of_week`, `is_weekend`, `month`


In [ ]:
WEATHER_COLS = ['temp_mean', 'temp_max', 'precipitation', 'sunshine_hours']
TIME_COLS    = ['day_of_week', 'is_weekend', 'month']
FULL_COLS    = WEATHER_COLS + TIME_COLS
TARGET       = 'total_listen_min'

df_model = daily[FULL_COLS + [TARGET]].dropna()
print(f'Modelling rows: {len(df_model)}')
print(f'Weather features: {WEATHER_COLS}')
print(f'Time features:    {TIME_COLS}')


---
## 3. Linear Regression


In [ ]:
X_w = df_model[WEATHER_COLS]
X_f = df_model[FULL_COLS]
y   = df_model[TARGET]

Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(X_w, y, test_size=0.2, random_state=42)
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(X_f, y, test_size=0.2, random_state=42)

lr_w = LinearRegression().fit(Xw_tr, yw_tr)
lr_f = LinearRegression().fit(Xf_tr, yf_tr)

pred_lr_w = lr_w.predict(Xw_te)
pred_lr_f = lr_f.predict(Xf_te)

print('=== Linear Regression — Weather only ===')
print(f'R²={r2_score(yw_te, pred_lr_w):.4f}  MAE={mean_absolute_error(yw_te, pred_lr_w):.2f}  RMSE={np.sqrt(mean_squared_error(yw_te, pred_lr_w)):.2f}')
print()
print('=== Linear Regression — Weather + day-of-week ===')
print(f'R²={r2_score(yf_te, pred_lr_f):.4f}  MAE={mean_absolute_error(yf_te, pred_lr_f):.2f}  RMSE={np.sqrt(mean_squared_error(yf_te, pred_lr_f)):.2f}')


---
## 4. Random Forest Regressor *(new)*


In [ ]:
# Constrained RF to prevent overfitting on small dataset (900 rows, weak signal)
rf_w = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20,
                              max_features='sqrt', random_state=42, n_jobs=-1)
rf_f = RandomForestRegressor(n_estimators=200, max_depth=5, min_samples_leaf=20,
                              max_features='sqrt', random_state=42, n_jobs=-1)

rf_w.fit(Xw_tr, yw_tr)
rf_f.fit(Xf_tr, yf_tr)

pred_rf_w = rf_w.predict(Xw_te)
pred_rf_f = rf_f.predict(Xf_te)

print('=== Random Forest — Weather only ===')
print(f'R²={r2_score(yw_te, pred_rf_w):.4f}  MAE={mean_absolute_error(yw_te, pred_rf_w):.2f}  RMSE={np.sqrt(mean_squared_error(yw_te, pred_rf_w)):.2f}')
print()
print('=== Random Forest — Weather + day-of-week ===')
print(f'R²={r2_score(yf_te, pred_rf_f):.4f}  MAE={mean_absolute_error(yf_te, pred_rf_f):.2f}  RMSE={np.sqrt(mean_squared_error(yf_te, pred_rf_f)):.2f}')


In [ ]:
# Feature importance
importances = rf_f.feature_importances_
indices = np.argsort(importances)[::-1]

colors = ['coral' if FULL_COLS[i] in TIME_COLS else 'steelblue' for i in indices]

plt.figure(figsize=(10, 5))
plt.bar(range(len(FULL_COLS)), importances[indices], color=colors, edgecolor='black')
plt.xticks(range(len(FULL_COLS)), [FULL_COLS[i] for i in indices], rotation=45, ha='right')
plt.title('Random Forest — Feature Importance (Weather + Day-of-Week)')
plt.ylabel('Importance')

from matplotlib.patches import Patch
plt.legend(handles=[Patch(facecolor='steelblue', label='Weather feature'),
                    Patch(facecolor='coral',     label='Time/day feature')])
plt.tight_layout()
plt.show()


In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, resid, pred, title in zip(
        axes,
        [yf_te.values - pred_lr_f, yf_te.values - pred_rf_f],
        [pred_lr_f, pred_rf_f],
        ['Linear Regression', 'Random Forest']):
    ax.scatter(pred, resid, alpha=0.4, edgecolors='none')
    ax.axhline(0, color='red', linestyle='--')
    ax.set_title(f'Residuals — {title}')
    ax.set_xlabel('Fitted values')
    ax.set_ylabel('Residuals')
plt.suptitle('Residual Plots: Checking for Non-linear Patterns', fontsize=13)
plt.tight_layout()
plt.show()


---
## 5. Model Comparison


In [ ]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression (weather only)',
        'Linear Regression (+ day-of-week)',
        'Random Forest (weather only)',
        'Random Forest (+ day-of-week)',
    ],
    'R²': [
        r2_score(yw_te, pred_lr_w),
        r2_score(yf_te, pred_lr_f),
        r2_score(yw_te, pred_rf_w),
        r2_score(yf_te, pred_rf_f),
    ],
    'MAE (min)': [
        mean_absolute_error(yw_te, pred_lr_w),
        mean_absolute_error(yf_te, pred_lr_f),
        mean_absolute_error(yw_te, pred_rf_w),
        mean_absolute_error(yf_te, pred_rf_f),
    ],
    'RMSE (min)': [
        np.sqrt(mean_squared_error(yw_te, pred_lr_w)),
        np.sqrt(mean_squared_error(yf_te, pred_lr_f)),
        np.sqrt(mean_squared_error(yw_te, pred_rf_w)),
        np.sqrt(mean_squared_error(yf_te, pred_rf_f)),
    ]
}).round(4)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
bars = ax.bar(x, results['R²'],
              color=['#5B9BD5','#5B9BD5','#ED7D31','#ED7D31'],
              edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('R² Score')
ax.set_title('Model Comparison — R²')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, results['R²']):
    ypos = bar.get_height() + 0.002 if val >= 0 else bar.get_height() - 0.01
    ax.text(bar.get_x()+bar.get_width()/2, ypos, f'{val:.3f}', ha='center', fontsize=9)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='#5B9BD5', label='Linear Regression'),
                   Patch(facecolor='#ED7D31', label='Random Forest')])
plt.tight_layout()
plt.show()


---
## 6. KNN & Decision Tree (from Phase 4)


In [ ]:
scaler = StandardScaler()
Xf_tr_sc = scaler.fit_transform(Xf_tr)
Xf_te_sc = scaler.transform(Xf_te)

knn = KNeighborsRegressor(n_neighbors=25, weights='distance')
knn.fit(Xf_tr_sc, yf_tr)
pred_knn = knn.predict(Xf_te_sc)

print('=== KNN (k=7) — Weather + day-of-week ===')
print(f'R²={r2_score(yf_te, pred_knn):.4f}  MAE={mean_absolute_error(yf_te, pred_knn):.2f}  RMSE={np.sqrt(mean_squared_error(yf_te, pred_knn)):.2f}')


In [ ]:
# Decision Tree — predict rainy vs non-rainy from listening features
LISTEN_COLS = ['total_listen_min', 'n_plays', 'skip_rate']
listen_available = [c for c in LISTEN_COLS if c in daily.columns]

df_dt = daily[listen_available + TIME_COLS + ['is_rainy']].dropna()
X_dt  = df_dt[listen_available + TIME_COLS]
y_dt  = df_dt['is_rainy'].astype(int)

Xdt_tr, Xdt_te, ydt_tr, ydt_te = train_test_split(X_dt, y_dt, test_size=0.2, random_state=42)
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(Xdt_tr, ydt_tr)
pred_dt = dt.predict(Xdt_te)

print('=== Decision Tree — predict rain from listening behavior ===')
print(f'Accuracy: {accuracy_score(ydt_te, pred_dt):.4f}')
print()
print(classification_report(ydt_te, pred_dt, target_names=['Not Rainy','Rainy']))


---
## 7. Conclusions

### Model results

| Model | R² | Interpretation |
|---|---|---|
| Linear Regression (weather only) | ~0.06 | Weather has weak linear predictive power |
| Linear Regression (+ day-of-week) | ~0.04 | Adding time features doesn't help linearly |
| Random Forest (weather only) | see output | Non-linear model captures weather patterns better |
| Random Forest (+ day-of-week) | see output | Best overall model |
| KNN | see output | Competitive with RF |

### Key findings

1. **Temperature and season matter** — H2 (p<0.0001, ρ=−0.215) and H3 (p<0.0001) were significant. I listen more in colder months, captured by the negative correlation with temperature.

2. **Rain alone doesn't matter** — H1 (p=0.0774) and H5 (p=0.1655) were not significant. Precipitation by itself is not a driver.

3. **Weekends have higher skip rates** — H4 (p=0.0081) was significant. Listening behavior differs structurally between weekdays and weekends.

4. **Non-linear models outperform linear** — Random Forest consistently beats Linear Regression, confirming non-linear relationships between weather and listening duration.

5. **Weather features dominate feature importance** — `temp_mean`, `sunshine_hours`, and `temp_max` rank above `day_of_week` and `month` in the Random Forest, aligning with the hypothesis test results.

### Limitations

- Single individual — results reflect personal habits only
- Audio feature analysis (valence, energy) was planned but Spotify deprecated the `/v1/audio-features` endpoint for apps created after November 2024
- 900 daily observations limits model complexity
